# Lecture 2: Python Foundations II

NumPy, Pandas, and exploratory data analysis using the public UCI Default of Credit Card Clients dataset.

## Learning objectives

Represent financial data with arrays and DataFrames. Audit the table. Use filtering, grouping, joins, transformations, correlations, and labelled charts. Interpret descriptive evidence without making causal claims.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = 'https://raw.githubusercontent.com/fkottas/Teaching-Courses/main/Machine%20Learning%20in%20Finance/2026-2027/lectures/lecture-00-01/data/credit/default_of_credit_card_clients.csv'
credit = pd.read_csv(DATA_URL)
credit = credit.rename(columns={'default payment next month': 'default_next_month'})
credit.head(3)

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default_next_month limit_band
0   1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1        low
1   2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1     medium
2   3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                   0        low

## NumPy calculation layer

In [2]:
values = np.array([100, 105, 102, 108])
prices = np.array([[100, 20], [105, 21], [102, 22], [108, 23]])
print(values.shape)
print(prices.shape)
print(prices.ndim)

(4,)
(4, 2)
2

In [3]:
prices = np.array([100, 105, 102, 108])
returns = prices[1:] / prices[:-1] - 1
np.round(returns, 4)

[ 0.05   -0.0286  0.0588]

In [4]:
asset_returns = np.array([0.08, 0.03, 0.01])
weights = np.array([0.50, 0.30, 0.20])
weighted = asset_returns * weights
print(np.round(weighted, 4))
print(round(weighted.sum(), 4))

[0.04  0.009 0.002]
0.051

In [5]:
x = np.array([10.0, 12.0, np.nan, 15.0])
print(np.mean(x))
print(np.nanmean(x))
print(np.isnan(x))

nan
12.333333333333334
[False False  True False]

## Load and audit the credit table

In [6]:
print(credit.shape)
print(credit.dtypes.value_counts())
print(credit.index.is_unique)

(30000, 25)
25    int64
dtype: int64
True

In [7]:
audit = pd.Series({'rows': len(credit), 'columns': credit.shape[1], 'missing_cells': int(credit.isna().sum().sum()), 'duplicate_rows': int(credit.duplicated().sum()), 'unique_ids': credit['ID'].nunique()})
audit

rows             30000
columns             25
missing_cells        0
duplicate_rows       0
unique_ids       30000

In [8]:
credit['default_next_month'].value_counts().rename({0: 'No default', 1: 'Default'})

No default    23364
Default        6636

## Descriptive statistics

In [9]:
numeric = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
summary = credit[numeric].describe().T
summary['IQR'] = summary['75%'] - summary['25%']
summary.round(2)

             count       mean        std       min       25%       50%       75%        max        IQR
LIMIT_BAL  30000.0  167484.32  129747.66   10000.0  50000.00  140000.0  240000.0  1000000.0  190000.00
AGE        30000.0      35.49       9.22      21.0     28.00      34.0      41.0       79.0      13.00
BILL_AMT1  30000.0   51223.33   73635.86 -165580.0   3558.75   22381.5   67091.0   964511.0   63532.25
PAY_AMT1   30000.0    5663.58   16563.28       0.0   1000.00    2100.0    5006.0   873552.0    4006.00

In [10]:
credit['limit_band'] = pd.cut(credit['LIMIT_BAL'], bins=[-np.inf, 100000, 300000, np.inf], labels=['low', 'medium', 'high'])
by_band = credit.groupby('limit_band', observed=False).agg(borrowers=('ID', 'size'), default_rate=('default_next_month', 'mean'), median_age=('AGE', 'median')).round(4)
by_band

            borrowers  default_rate  median_age
limit_band                                     
low             12498        0.2948        32.0
medium          12939        0.1814        34.0
high             4563        0.1326        36.0

In [11]:
credit['age_band'] = pd.cut(credit['AGE'], bins=[20, 30, 40, 50, 80], labels=['21-30', '31-40', '41-50', '51-79'])
pd.pivot_table(credit, index='age_band', columns='limit_band', values='default_next_month', aggfunc='mean').round(3)

limit_band    low  medium   high
age_band
21-30       0.340   0.213  0.170
31-40       0.292   0.179  0.127
41-50       0.276   0.164  0.116
51-79       0.259   0.156  0.104

## Labels, joins, and derived variables

In [12]:
education_lookup = pd.DataFrame({'EDUCATION': [1, 2, 3, 4], 'education_label': ['graduate school', 'university', 'high school', 'others']})
credit.merge(education_lookup, on='EDUCATION', how='left')[['EDUCATION', 'education_label']].drop_duplicates().sort_values('EDUCATION')

EDUCATION    education_label
1            graduate school
2            university
3            high school
4            others

In [13]:
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
payment_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
credit = credit.assign(total_bill=credit[bill_cols].sum(axis=1), total_payment=credit[payment_cols].sum(axis=1))
credit['payment_to_limit'] = credit['total_payment'] / credit['LIMIT_BAL']
credit[['total_bill', 'total_payment', 'payment_to_limit']].describe().round(3)

count       30000.000
mean       ...
median     ...

## Visual evidence

In [14]:
credit['default_next_month'].value_counts().sort_index().plot(kind='bar', color=['#2D9DB0', '#D77B6D'])
plt.title('Borrowers by target class')
plt.xlabel('Target: 0 = no default, 1 = default')
plt.ylabel('Borrowers')
plt.show()

In [15]:
by_band['default_rate'].plot(kind='bar', color='#2D9DB0')
plt.title('Default rate by credit-limit band')
plt.xlabel('Credit-limit band')
plt.ylabel('Share of borrowers')
plt.xticks(rotation=0)
plt.show()

In [16]:
sample = credit.sample(1200, random_state=42)
plt.scatter(sample['LIMIT_BAL'], sample['BILL_AMT1'], s=10, alpha=0.35, color='#2D9DB0')
plt.title('Credit limit and first bill amount')
plt.xlabel('Credit limit (New Taiwan dollars)')
plt.ylabel('First bill amount (New Taiwan dollars)')
plt.show()

In [17]:
credit[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1', 'default_next_month']].corr().round(3)

                    LIMIT_BAL    AGE  BILL_AMT1  PAY_AMT1  default_next_month
LIMIT_BAL               1.000  0.145      0.285     0.195              -0.154
AGE                     0.145  1.000      0.056     0.026               0.014
BILL_AMT1               0.285  0.056      1.000     0.140              -0.020
PAY_AMT1                0.195  0.026      0.140     1.000              -0.073
default_next_month     -0.154  0.014     -0.020    -0.073               1.000

## Interpretation and limitations

The supplied sample shows a higher default rate among borrowers in the low-limit band. This is descriptive association. It may reflect repayment history, borrower composition, underwriting policy, and other variables. Week 3 will prepare variables without leakage before supervised models are introduced.

In [18]:
def audit_table(df, target):
    return {'shape': df.shape, 'missing_cells': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'target_counts': df[target].value_counts().to_dict()}

audit_table(credit, 'default_next_month')

{'shape': (30000, 25), 'missing_cells': 0, 'duplicate_rows': 0, 'target_counts': {0: 23364, 1: 6636}}

## Submission checklist

Include the source, access date, observation unit, target definition, data dictionary, audit, descriptive statistics, one group comparison, two labelled charts, interpretations, and limitations. Submit the notebook one day before the next course meeting.